
Рабочий процесс:
- загрузка набора данных с `image_path`/`abc_path` или готовыми полями `image`/`transcription`
- расширение токенизатора текстовыми маркерами
- чистая загрузка весов LEGATO
- замораживание большей части модели для небольшого теста
- запуск взвешенной контролируемой тонкой настройки

In [ ]:
from huggingface_hub import login
login(token="")

In [9]:
from __future__ import annotations

import json
import os
import string
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
import torch.distributed as dist
from datasets import Dataset, DatasetDict, load_from_disk
from huggingface_hub import login, snapshot_download
from PIL import Image
from safetensors.torch import load_file as load_safetensors_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoProcessor, Seq2SeqTrainingArguments, set_seed
from transformers import MllamaVisionModel


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'legato').exists() and (candidate / 'scripts' / 'train.py').exists():
            return candidate
    raise FileNotFoundError('Не удалось найти корень проекта LEGATO.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Важно: импортируем legato.models до AutoProcessor/AutoModel.from_pretrained,
# чтобы сработала регистрация кастомных классов.
from legato.metrics import compute_error_rates
from legato.models import LegatoModel, LegatoConfig
from legato.trainer import LegatoTrainer

NOTEBOOK_ROOT = PROJECT_ROOT / 'legato-utils-new'
ARTIFACT_ROOT = NOTEBOOK_ROOT / 'artifacts'
OUTPUT_ROOT = NOTEBOOK_ROOT / 'outputs'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('NOTEBOOK_ROOT =', NOTEBOOK_ROOT)


def load_repo_state_dict(repo_id: str, token: str | None = None) -> dict[str, torch.Tensor]:
    snapshot_path = Path(snapshot_download(repo_id=repo_id, token=token))
    safetensors_files = sorted(snapshot_path.glob('*.safetensors'))
    if not safetensors_files:
        raise FileNotFoundError(f'В snapshot не найдено *.safetensors: {snapshot_path}')
    merged = {}
    for safetensors_path in safetensors_files:
        merged.update(load_safetensors_file(str(safetensors_path)))
    return merged


def remap_legato_keys_for_current_mllama(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    remapped = {}
    for key, value in state_dict.items():
        new_key = key
        if key.startswith('language_model.model.'):
            new_key = 'model.language_model.' + key[len('language_model.model.'):]
        elif key.startswith('language_model.lm_head.'):
            new_key = 'lm_head.' + key[len('language_model.lm_head.'):]
        elif key.startswith('language_model.'):
            new_key = 'model.' + key
        elif key.startswith('multi_modal_projector.'):
            new_key = 'model.multi_modal_projector.' + key[len('multi_modal_projector.'):]
        elif key.startswith('vision_model.'):
            # Не грузим vision из LEGATO-чекпоинта: ниже загрузим encoder отдельно из base vision model.
            continue
        remapped[new_key] = value
    return remapped


def load_legato_model_clean(model_ref: str, token: str | None = None):
    config = AutoConfig.from_pretrained(model_ref, token=token)
    assert isinstance(config, LegatoConfig), f'Ожидался LegatoConfig, получили: {type(config)}'
    model = LegatoModel(config, load_pretrained_encoder=False)

    raw_state_dict = load_repo_state_dict(model_ref, token=token)
    remapped_state_dict = remap_legato_keys_for_current_mllama(raw_state_dict)
    incompatible = model.load_state_dict(remapped_state_dict, strict=False)

    encoder_ref = getattr(config, 'encoder_pretrained_model_name_or_path', None)
    if encoder_ref is None:
        raise ValueError('В конфиге отсутствует encoder_pretrained_model_name_or_path')
    model.model.vision_model = MllamaVisionModel.from_pretrained(encoder_ref, token=token)
    model.config.vision_config = model.model.vision_model.config
    for param in model.model.vision_model.parameters():
        param.requires_grad = False

    print('missing keys:', len(incompatible.missing_keys))
    print('unexpected keys:', len(incompatible.unexpected_keys))
    if incompatible.missing_keys:
        print('Первые missing keys:', incompatible.missing_keys[:20])
    if incompatible.unexpected_keys:
        print('Первые unexpected keys:', incompatible.unexpected_keys[:20])

    return model


PROJECT_ROOT = F:\legato
NOTEBOOK_ROOT = F:\legato\legato-utils-new


In [10]:
@dataclass
class PipelineConfig:
    base_model: str = 'guangyangmusic/legato'
    dataset_path: str = r'F:\datasets\PDMX\dataset_root\hf_dataset' # Путь для датасета
    output_name: str = 'legato-text-music-portable'
    seed: int = 42
    abc_encoding: str = 'utf-8'
    markers: tuple[str, ...] = ('<TITLE>', '<LYRICS>', '<ANNOT>')
    force_add_characters: tuple[str, ...] = tuple(string.ascii_letters + string.digits)
    music_loss_weight: float = 0.25
    train_split_name: str = 'train'
    val_split_name: str = 'validation'
    test_split_name: str = 'test'
    max_train_samples: int = 8
    max_val_samples: int = 4
    max_test_samples: int = 1
    num_train_epochs: int = 1
    max_steps: int = 5
    learning_rate: float = 5e-5
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 1
    dataloader_num_workers: int = 0
    generation_max_length: int = 1024
    generation_num_beams: int = 1
    logging_steps: int = 1
    eval_steps: int = 1
    save_steps: int = 1
    train_last_n_decoder_layers: int = 1
    train_lm_head: bool = True
    train_token_embeddings: bool = True
    run_train: bool = False
    run_eval: bool = False


CFG = PipelineConfig()
set_seed(CFG.seed)

DATASET_PATH = CFG.dataset_path
RUN_ROOT = OUTPUT_ROOT / CFG.output_name
PROCESSOR_OUT = RUN_ROOT / 'processor_extended'
MODEL_OUT = RUN_ROOT / 'model'
REPORT_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_vocab_report.json'
META_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_tokenizer_meta.json'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSOR_OUT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)

print(CFG)


PipelineConfig(base_model='guangyangmusic/legato', dataset_path='F:\\datasets\\PDMX\\dataset_root\\hf_dataset', output_name='legato-text-music-portable', seed=42, abc_encoding='utf-8', markers=('<TITLE>', '<LYRICS>', '<ANNOT>'), force_add_characters=('a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'), music_loss_weight=0.25, train_split_name='train', val_split_name='validation', test_split_name='test', max_train_samples=8, max_val_samples=4, max_test_samples=1, num_train_epochs=1, max_steps=5, learning_rate=5e-05, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=1, dataloader_num_workers=0, generation_max_length=1024, generation_num_beams=1, logging_steps=1, eval_steps=1, save_steps=1, train_

## 1. Нормализация датасета

In [11]:
def resolve_existing_path(raw_path: str | Path, base_dir: Path) -> Path:
    candidate = Path(raw_path)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    variants = [base_dir / candidate, PROJECT_ROOT / candidate, candidate]
    for path in variants:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'Не найден путь: {raw_path}')


def build_transcription_from_abc(abc_text: str) -> str:
    return abc_text


def load_image_from_row(row, base_dir: Path):
    if 'image' in row and row['image'] is not None:
        image_value = row['image']
        if isinstance(image_value, str):
            image_file = resolve_existing_path(image_value, base_dir)
            return Image.open(image_file).convert('RGB'), str(image_file)
        return image_value.convert('RGB') if hasattr(image_value, 'convert') else image_value, None

    for key in ['image_path', 'img_path', 'path', 'image_file', 'image_filepath']:
        if key in row and row[key] is not None:
            image_file = resolve_existing_path(row[key], base_dir)
            return Image.open(image_file).convert('RGB'), str(image_file)

    raise KeyError(f'Не найдено поле с изображением. Доступные поля: {list(row.keys())}')


def load_transcription_from_row(row, base_dir: Path):
    if 'transcription' in row and row['transcription'] is not None:
        return str(row['transcription']), None

    if 'abc' in row and row['abc'] is not None:
        return build_transcription_from_abc(str(row['abc'])), None

    for key in ['abc_path', 'transcription_path', 'label_path', 'abc_file', 'annotation_path']:
        if key in row and row[key] is not None:
            abc_file = resolve_existing_path(row[key], base_dir)
            return build_transcription_from_abc(abc_file.read_text(encoding=CFG.abc_encoding)), str(abc_file)

    raise KeyError(f'Не найдено поле с ABC/транскрипцией. Доступные поля: {list(row.keys())}')


def normalize_split(split_ds, split_name: str):
    def convert_row(row):
        image, image_path_resolved = load_image_from_row(row, DATASET_PATH)
        transcription, abc_path_resolved = load_transcription_from_row(row, DATASET_PATH)
        return {
            'id': row.get('id', ''),
            'image': image,
            'transcription': transcription,
            'image_path_resolved': image_path_resolved,
            'abc_path_resolved': abc_path_resolved,
            'split_name': split_name,
        }

    return Dataset.from_list([convert_row(split_ds[i]) for i in range(len(split_ds))])


raw_dataset = load_from_disk(str(DATASET_PATH))
assert isinstance(raw_dataset, DatasetDict), 'Ожидался DatasetDict с train/validation/test.'

print('Найденные сплиты:', list(raw_dataset.keys()))

train_raw = raw_dataset[CFG.train_split_name].select(range(min(CFG.max_train_samples, len(raw_dataset[CFG.train_split_name]))))
val_raw = raw_dataset[CFG.val_split_name].select(range(min(CFG.max_val_samples, len(raw_dataset[CFG.val_split_name]))))
test_raw = raw_dataset[CFG.test_split_name].select(range(min(CFG.max_test_samples, len(raw_dataset[CFG.test_split_name]))))

dataset = DatasetDict({
    'train': normalize_split(train_raw, 'train'),
    'val': normalize_split(val_raw, 'val'),
    'test': normalize_split(test_raw, 'test'),
})

for split_name, split_ds in dataset.items():
    print(split_name, len(split_ds), split_ds.column_names)


Найденные сплиты: ['train', 'validation', 'test']
train 8 ['id', 'image', 'transcription', 'image_path_resolved', 'abc_path_resolved', 'split_name']
val 4 ['id', 'image', 'transcription', 'image_path_resolved', 'abc_path_resolved', 'split_name']
test 1 ['id', 'image', 'transcription', 'image_path_resolved', 'abc_path_resolved', 'split_name']


## 2. Анализ словаря и расширение tokenizer

In [12]:
base_processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
base_tokenizer = base_processor.tokenizer


def iter_transcriptions(ds_dict: DatasetDict) -> Iterable[str]:
    for split_ds in ds_dict.values():
        for item in split_ds['transcription']:
            if item is not None:
                yield item


def is_single_token(tokenizer, text: str) -> bool:
    return len(tokenizer(text, add_special_tokens=False)['input_ids']) == 1


all_transcriptions = list(iter_transcriptions(dataset))
char_counter = Counter(''.join(all_transcriptions))
unique_chars = sorted(char_counter)
forced_chars = sorted(set(CFG.force_add_characters))

dataset_chars_not_single = sorted(ch for ch in unique_chars if not is_single_token(base_tokenizer, ch))
forced_chars_not_single = sorted(ch for ch in forced_chars if not is_single_token(base_tokenizer, ch))
chars_to_add = sorted(set(dataset_chars_not_single) | set(forced_chars_not_single))

report = {
    'base_model': CFG.base_model,
    'dataset_path': str(DATASET_PATH),
    'num_train_items': len(dataset['train']),
    'num_val_items': len(dataset['val']),
    'num_test_items': len(dataset['test']),
    'num_transcriptions_seen': len(all_transcriptions),
    'markers': list(CFG.markers),
    'chars_to_add': chars_to_add,
    'top_100_char_counts': dict(char_counter.most_common(100)),
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
tokenizer = processor.tokenizer
original_vocab_size = len(tokenizer)

existing_vocab = tokenizer.get_vocab()
marker_tokens_to_add = [tok for tok in CFG.markers if tok not in existing_vocab]
char_tokens_to_add = [tok for tok in chars_to_add if tok not in existing_vocab]

num_added_markers = tokenizer.add_special_tokens({'additional_special_tokens': marker_tokens_to_add}) if marker_tokens_to_add else 0
num_added_chars = tokenizer.add_tokens(char_tokens_to_add) if char_tokens_to_add else 0
processor.save_pretrained(PROCESSOR_OUT)

tokenizer_meta = {
    'base_model': CFG.base_model,
    'processor_out': str(PROCESSOR_OUT),
    'original_vocab_size': original_vocab_size,
    'extended_vocab_size': len(tokenizer),
    'num_added_markers': num_added_markers,
    'num_added_chars': num_added_chars,
    'marker_tokens': list(CFG.markers),
    'marker_token_ids': [tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers],
    'char_tokens_added': char_tokens_to_add,
}
META_PATH.write_text(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False))


{
  "base_model": "guangyangmusic/legato",
  "processor_out": "F:\\legato\\legato-utils-new\\outputs\\legato-text-music-portable\\processor_extended",
  "original_vocab_size": 4097,
  "extended_vocab_size": 4100,
  "num_added_markers": 3,
  "num_added_chars": 0,
  "marker_tokens": [
    "<TITLE>",
    "<LYRICS>",
    "<ANNOT>"
  ],
  "marker_token_ids": [
    4097,
    4098,
    4099
  ],
  "char_tokens_added": []
}


## 3. Взвешенный trainer

In [13]:
def build_token_weight_vector(vocab_size: int, downweighted_token_ids: list[int], default_weight: float, downweight: float) -> torch.Tensor:
    weights = torch.full((vocab_size,), float(default_weight), dtype=torch.float32)
    if downweighted_token_ids:
        weights[torch.tensor(downweighted_token_ids, dtype=torch.long)] = float(downweight)
    return weights


special_ids = set(tokenizer.all_special_ids)
downweighted_token_ids = [idx for idx in range(original_vocab_size) if idx not in special_ids]
token_weight_vector = build_token_weight_vector(
    vocab_size=len(tokenizer),
    downweighted_token_ids=downweighted_token_ids,
    default_weight=1.0,
    downweight=CFG.music_loss_weight,
)


class WeightedLegatoTrainer(LegatoTrainer):
    def __init__(self, *args, token_weight_vector: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self._token_weight_vector = token_weight_vector.clone().float()
        self._token_weight_vector_device = None

    def _get_token_weight_vector(self, device: torch.device) -> torch.Tensor:
        if self._token_weight_vector_device != device:
            self._token_weight_vector = self._token_weight_vector.to(device)
            self._token_weight_vector_device = device
        return self._token_weight_vector

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = {k: v for k, v in inputs.items() if not k.startswith('gen_')}
        labels = inputs['labels']
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs, use_cache=False)
        logits = outputs.logits.float()

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view_as(shift_labels)

        weight_vector = self._get_token_weight_vector(shift_logits.device)
        safe_labels = shift_labels.clamp(min=0)
        token_weights = weight_vector[safe_labels]
        token_weights = torch.where(shift_labels.eq(-100), torch.zeros_like(token_weights), token_weights)

        weighted_loss = (token_losses * token_weights).sum() / token_weights.sum().clamp_min(1e-8)
        return (weighted_loss, outputs) if return_outputs else weighted_loss


## 4. Загрузка модели и сильная заморозка

In [14]:
model = load_legato_model_clean(CFG.base_model, token=HF_TOKEN)
model.resize_token_embeddings(len(tokenizer))


def freeze_all_parameters(model):
    for param in model.parameters():
        param.requires_grad = False


def get_decoder_layers(model):
    candidates = [
        ('model.language_model.layers', getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None)),
        ('model.language_model.model.layers', getattr(getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'model', None), 'layers', None)),
        ('language_model.layers', getattr(getattr(model, 'language_model', None), 'layers', None)),
        ('language_model.model.layers', getattr(getattr(getattr(model, 'language_model', None), 'model', None), 'layers', None)),
    ]
    for name, layers in candidates:
        if layers is not None:
            return name, layers
    raise AttributeError('Не удалось найти слои декодера.')


freeze_all_parameters(model)
for param in model.model.vision_model.parameters():
    param.requires_grad = False

decoder_path, decoder_layers = get_decoder_layers(model)
for layer in list(decoder_layers)[-CFG.train_last_n_decoder_layers:]:
    for param in layer.parameters():
        param.requires_grad = True

if CFG.train_token_embeddings:
    for param in model.get_input_embeddings().parameters():
        param.requires_grad = True

if CFG.train_lm_head:
    for param in model.get_output_embeddings().parameters():
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print('Путь к слоям декодера:', decoder_path)
print('Обучаемые параметры:', f'{trainable_params:,}')
print('Замороженные параметры:', f'{frozen_params:,}')


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


missing keys: 0
unexpected keys: 0
Путь к слоям декодера: model.language_model.layers
Обучаемые параметры: 11,584,512
Замороженные параметры: 932,065,819


## 5. Trainer и tiny-конфиг

In [15]:
def get_metric_target(examples):
    return {
        'label_ids': tokenizer(
            examples['transcription'],
            add_special_tokens=False,
            truncation=True,
            max_length=CFG.generation_max_length,
        )['input_ids'],
    }


map_num_proc = CFG.dataloader_num_workers if CFG.dataloader_num_workers > 0 else None

metric_targets = dataset['val'].map(
    get_metric_target,
    remove_columns=dataset['val'].column_names,
    num_proc=map_num_proc,
    batched=True,
).to_dict()
tokens_to_mask = torch.tensor([tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers] + [tokenizer.pad_token_id])


def collate_fn(examples):
    outputs = processor(
        images=[example['image'] for example in examples],
        text=[example['transcription'] for example in examples],
        return_num_tiles=True,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    gen_outputs = processor(
        num_tiles=outputs.pop('num_tiles'),
        truncation=True,
        padding=True,
        return_tensors='pt',
    )
    outputs.update({
        f'gen_{k}': outputs[k] if k not in gen_outputs else gen_outputs[k]
        for k in outputs
    })
    outputs['labels'] = outputs['input_ids'].clone().masked_fill(torch.isin(outputs['input_ids'], tokens_to_mask), -100)
    return outputs


special_tokens = [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, -100]


def remove_special_tokens(array):
    masks = np.isin(array, special_tokens, invert=True)
    return [a[mask] for a, mask in zip(array, masks)]


def metric_fn(p):
    preds = remove_special_tokens(p.predictions)
    metric_num_workers = max(1, CFG.dataloader_num_workers)
    results = [
        compute_error_rates(
            tokenizer,
            metric_num_workers,
            *metric_targets.values(),
            preds,
        )
    ] if getattr(training_args, 'process_index', 0) == 0 else [None]

    if dist.is_available() and dist.is_initialized():
        dist.broadcast_object_list(results, src=0)

    return results[0]



training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_OUT),
    remove_unused_columns=False,
    do_train=True,
    do_eval=True,
    predict_with_generate=True,
    metric_for_best_model='eval_SER',
    greater_is_better=False,
    load_best_model_at_end=False,
    num_train_epochs=CFG.num_train_epochs,
    max_steps=CFG.max_steps,
    learning_rate=CFG.learning_rate,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    dataloader_num_workers=CFG.dataloader_num_workers,
    logging_steps=CFG.logging_steps,
    save_strategy='steps',
    save_steps=CFG.save_steps,
    eval_strategy='steps',
    eval_steps=CFG.eval_steps,
    generation_max_length=CFG.generation_max_length,
    generation_num_beams=CFG.generation_num_beams,
    bf16=torch.cuda.is_available(),
    bf16_full_eval=torch.cuda.is_available(),
    report_to='none',
    seed=CFG.seed,
    logging_dir=str(MODEL_OUT / 'logs'),
)

trainer = WeightedLegatoTrainer(
    model=model,
    args=training_args,
    token_weight_vector=token_weight_vector,
    data_collator=collate_fn,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    compute_metrics=metric_fn,
)

print('Trainer готов.')


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Trainer готов.


## 6. Запуск

In [16]:
RUN_TRAIN = True
RUN_EVAL = True

if RUN_TRAIN:
    train_result = trainer.train()
    trainer.save_model(str(MODEL_OUT / 'final'))
    processor.save_pretrained(MODEL_OUT / 'final')
    with open(MODEL_OUT / 'train_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(train_result.metrics, f, indent=2)
    print('Обучение завершено.')

if RUN_EVAL:
    eval_metrics = trainer.evaluate()
    with open(MODEL_OUT / 'eval_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(eval_metrics, f, indent=2)
    print(eval_metrics)


Step,Training Loss,Validation Loss,Ler,Cer,Ser
1,2.960000,4.652186,78.571429,83.119548,83.389687
2,6.349500,4.637464,78.571429,83.683940,83.389687
3,6.141300,4.626758,78.571429,83.735249,83.643280
4,5.799600,4.620750,78.571429,84.761416,83.727811
5,1.338300,4.617033,78.571429,83.735249,83.643280


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
computing LER...: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.44it/s]


Обучение завершено.


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.44it/s]

{'eval_loss': 4.617204666137695, 'eval_LER': 78.57142857142857, 'eval_CER': 84.29964084145716, 'eval_SER': 83.68554522400676, 'eval_runtime': 97.1071, 'eval_samples_per_second': 0.041, 'eval_steps_per_second': 0.041, 'epoch': 0.625}


Большое количество warning при перезагрузке лучшего чекпоинта из-за того, что vision_model не сохраняется в checkpoint

это неприятно, но не означает, что сам train сломан

это можно игнорировать или просто выключить load_best_model_at_end

In [17]:
def preview_predictions(num_examples: int = 1):
    subset = dataset['val'].select(range(min(num_examples, len(dataset['val']))))
    outputs = trainer.predict(subset)

    preds = outputs.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    decoded = processor.batch_decode(preds, skip_special_tokens=True)

    for idx, sample in enumerate(decoded):
        print('=' * 80)
        print(f'Пример {idx}')
        print(sample[:4000])
        print('\nGT:')
        print(subset[idx]['transcription'][:4000])


preview_predictions(1)


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.72s/it]


Пример 0
X:1
T:<|text|>
T:<|text|>
L:1/8
Q:1/4=108
M:4/4
I:linebreak $
K:C
V:1 treble
V:1
 [GA]2 [GA]2 [Ac]2 [Ac]2 | [Ac]2 [Ac]2 [GB]2 ([AA][GA]) | [FA]2 [FA]2 ([GG]2 [AA]2) | [GA]2 [GA]2 ([GA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | %5
 [FA]2 [FA]2 ([GG]2 [AA]2) | [GA]2 [GA]2 ([GA][GA]) [GA]2 |$ [GA]2 [GA]2 ([GA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [FA]2 [FA]2 ([GG]2 [AA]2) | %10
 [GA]2 [GA]2 ([GA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 |$ [GA]2 [GB]2 ([AA][GA]) [GA]2 | %15
 [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | %20
 [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 |$ [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | %25
 [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2 | [GA]2 [GB]2 ([AA][GA]) [GA]2